In [1]:
%pip install datasets transformers evaluate jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 70.6 MB/s eta 0:00:00:00:01
Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
import datasets
import evaluate
import numpy as np
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Union
from datasets import load_dataset, Audio, Dataset, DatasetDict
#from transformers import Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor, Wav2Vec2Processor, AutoModelForCTC, TrainingArguments, Trainer

In [4]:
dataset = datasets.load_dataset("mozilla-foundation/common_voice_11_0", "as")

The repository for mozilla-foundation/common_voice_11_0 contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/mozilla-foundation/common_voice_11_0.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N]  y


n_shards.json:   0%|          | 0.00/12.2k [00:00<?, ?B/s]

as_train_0.tar:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

as_dev_0.tar:   0%|          | 0.00/16.3M [00:00<?, ?B/s]

as_test_0.tar:   0%|          | 0.00/12.1M [00:00<?, ?B/s]

...


Reading metadata...: 308it [00:00, 86671.96it/s]


Generating other split: 0 examples [00:00, ? examples/s]


Reading metadata...: 297it [00:00, 89017.31it/s]


Generating invalidated split: 0 examples [00:00, ? examples/s]


Reading metadata...: 163it [00:00, 68292.03it/s]


In [5]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['client_id', 'path', 'audio', 'sentence', 'up_votes', 'down_votes', 'age', 'gender', 'accent', 'locale', 'segment'],
        num_rows: 824
    })
    validation: Dataset({
        features: ['client_id', 'path', 'audio', 'sentence', 'up_votes', 'down_votes', 'age', 'gender', 'accent', 'locale', 'segment'],
        num_rows: 469
    })
    test: Dataset({
        features: ['client_id', 'path', 'audio', 'sentence', 'up_votes', 'down_votes', 'age', 'gender', 'accent', 'locale', 'segment'],
        num_rows: 308
    })
    other: Dataset({
        features: ['client_id', 'path', 'audio', 'sentence', 'up_votes', 'down_votes', 'age', 'gender', 'accent', 'locale', 'segment'],
        num_rows: 297
    })
    invalidated: Dataset({
        features: ['client_id', 'path', 'audio', 'sentence', 'up_votes', 'down_votes', 'age', 'gender', 'accent', 'locale', 'segment'],
        num_rows: 163
    })
})


In [6]:
#removing unnecessary columns
dataset=dataset.remove_columns(["accent", "age", "client_id", "down_votes", "gender", "locale", "path", "segment", "up_votes"])

In [7]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['audio', 'sentence'],
        num_rows: 824
    })
    validation: Dataset({
        features: ['audio', 'sentence'],
        num_rows: 469
    })
    test: Dataset({
        features: ['audio', 'sentence'],
        num_rows: 308
    })
    other: Dataset({
        features: ['audio', 'sentence'],
        num_rows: 297
    })
    invalidated: Dataset({
        features: ['audio', 'sentence'],
        num_rows: 163
    })
})


In [8]:
from datasets import Audio
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

In [9]:
print(dataset["train"][0])

{'audio': {'path': '/root/.cache/huggingface/datasets/downloads/extracted/fdcfd174c1db561f74a5aab292ff32458ceffd67c10de1ac5f5b77eae211090c/as_train_0/common_voice_as_22074894.mp3', 'array': array([ 5.29395592e-23, -6.61744490e-23,  1.48892510e-22, ...,
        1.20360312e-07, -1.29233990e-06, -1.51768404e-06]), 'sampling_rate': 16000}, 'sentence': 'দেখিলে যে অসমীয়া মানুহৰ জ্ঞান-উন্নতি পিনে অলপাে মনকাণ নাই'}


In [10]:
from transformers import WhisperFeatureExtractor
feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-tiny")

from transformers import WhisperTokenizer
tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-tiny", language="assamese", task="transcribe")

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/494k [00:00<?, ?B/s]

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

In [11]:
def prepare_dataset(batch):
    # load and resample audio data from 48 to 16kHz
    audio = batch["audio"]
    # compute log-Mel input features from input audio array
    batch["input_features"] = feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]
    # encode target text to label ids
    batch["labels"] = tokenizer(batch["sentence"]).input_ids
    return batch

In [12]:
dataset = dataset.map(prepare_dataset, remove_columns=dataset.column_names["train"], num_proc=4)

/opt/conda/lib/python3.10/site-packages/multiprocess/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()


Map (num_proc=4):   0%|          | 0/824 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/469 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/308 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/297 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/163 [00:00<?, ? examples/s]

In [13]:
from transformers import WhisperProcessor
processor = WhisperProcessor.from_pretrained("openai/whisper-tiny", language="assamese", task="transcribe")

import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union

# put together a list of samples into a mini training batch, https://www.youtube.com/watch?v=-RPeakdlHYo

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lengths and need different padding methods
        # first treat the audio inputs by simply returning torch tensors
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # get the tokenized label sequences
        label_features = [{"input_ids": feature["labels"]} for feature in features]

        # pad the labels to max length
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # if bos token is appended in previous tokenization step,
        # cut bos token here as it's append later anyways
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch


In [15]:
data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

In [16]:
import evaluate
metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    # replace -100 with the pad_token_id
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    # we do not want to group tokens when computing the metrics
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    wer = metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

## Model 1: learning rate: 1e-5, gradient acculumation steps = 1, num_train_epochs=10

In [25]:
from transformers import WhisperForConditionalGeneration

# Set device

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny").to(device)
#model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny")

model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

from transformers import Seq2SeqTrainingArguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-tiny-assamese",  # change to a repo name of your choice
    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,  # increase by 2x for every 2x decrease in batch size
    learning_rate=1e-5,
    warmup_steps=500,
    num_train_epochs=10,
    #max_steps=4000,
    gradient_checkpointing=True,
    fp16=True,
    evaluation_strategy="steps",
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=500,
    eval_steps=50,
    logging_steps=50,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=False,
)

from transformers import Seq2SeqTrainer
trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
)

/opt/conda/lib/python3.10/site-packages/transformers/training_args.py:1545: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [26]:
trainer.train()

/opt/conda/lib/python3.10/site-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[attr-defined]


Step,Training Loss,Validation Loss,Wer
50,3.243900,2.578773,1.464344
100,2.230900,1.826738,2.057787
150,1.620700,1.506765,4.050000
200,1.427100,1.404074,2.530738
250,1.336800,1.346907,2.426639
300,1.268300,1.302943,1.974590
350,1.201600,1.253020,1.711475
400,1.129300,1.187052,1.640984
450,0.967000,1.025061,1.378689
500,0.694300,0.773586,1.345492


/opt/conda/lib/python3.10/site-packages/transformers/modeling_utils.py:2618: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 448, 'suppress_tokens': [], 'begin_suppress_tokens': [220, 50257]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[attr-defined]
There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


TrainOutput(global_step=520, training_loss=1.4747935881981482, metrics={'train_runtime': 3695.441, 'train_samples_per_second': 2.23, 'train_steps_per_second': 0.141, 'total_flos': 2.028596133888e+17, 'train_loss': 1.4747935881981482, 'epoch': 10.0})

In [30]:
trainer.save_model("/kaggle/working/finetune_model_1")

In [28]:
trainer.evaluate()

{'eval_loss': 0.773586094379425,
 'eval_wer': 1.3610655737704918,
 'eval_runtime': 185.0545,
 'eval_samples_per_second': 1.664,
 'eval_steps_per_second': 0.211,
 'epoch': 10.0}

In [31]:
!zip -r /kaggle/working/finetune_model_1.zip /kaggle/working/finetune_model_1/

/opt/conda/lib/python3.10/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


  adding: kaggle/working/finetune_model_1/ (stored 0%)
  adding: kaggle/working/finetune_model_1/training_args.bin (deflated 51%)
  adding: kaggle/working/finetune_model_1/config.json (deflated 59%)
  adding: kaggle/working/finetune_model_1/model.safetensors (deflated 8%)
  adding: kaggle/working/finetune_model_1/generation_config.json (deflated 73%)
  adding: kaggle/working/finetune_model_1/preprocessor_config.json (deflated 42%)


## Model 2: learning rate: 1e-5, gradient acculumation steps = 1, num_train_epochs=20

In [32]:
from transformers import WhisperForConditionalGeneration

# Set device

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model2 = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny").to(device)
#model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny")
model2.config.forced_decoder_ids = None
model2.config.suppress_tokens = []

from transformers import Seq2SeqTrainingArguments
training_args2 = Seq2SeqTrainingArguments(
    output_dir="./whisper-tiny-assamese",  # change to a repo name of your choice
    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,  # increase by 2x for every 2x decrease in batch size
    learning_rate=1e-5,
    warmup_steps=500,
    num_train_epochs=20,
    weight_decay=0.005,
    #max_steps=4000,
    gradient_checkpointing=True,
    fp16=True,
    evaluation_strategy="steps",
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=500,
    eval_steps=50,
    logging_steps=50,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=False,
)

from transformers import Seq2SeqTrainer

trainer2 = Seq2SeqTrainer(
    args=training_args2,
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
)

/opt/conda/lib/python3.10/site-packages/transformers/training_args.py:1545: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [33]:
trainer2.train()
trainer2.save_model("/kaggle/working/finetune_model_2")
!zip -r /kaggle/working/finetune_model_2.zip /kaggle/working/finetune_model_2/

/opt/conda/lib/python3.10/site-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[attr-defined]


Step,Training Loss,Validation Loss,Wer
50,0.516800,0.767207,1.388525
100,0.492200,0.744848,1.265984
150,0.465900,0.719753,1.152049
200,0.426300,0.693789,1.266803
250,0.382700,0.664200,1.260656
300,0.346800,0.657176,1.139344
350,0.299400,0.622213,1.029508
400,0.264300,0.640157,1.181148
450,0.220000,0.645737,1.008197
500,0.181000,0.631808,1.229098


/opt/conda/lib/python3.10/site-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[attr-defined]
/opt/conda/lib/python3.10/site-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[attr-defined]
There were missing keys in the checkpoint model loaded: ['proj_out.weight'].
/opt/conda/lib/python3.10/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


  adding: kaggle/working/finetune_model_2/ (stored 0%)
  adding: kaggle/working/finetune_model_2/training_args.bin (deflated 51%)
  adding: kaggle/working/finetune_model_2/config.json (deflated 59%)
  adding: kaggle/working/finetune_model_2/model.safetensors (deflated 8%)
  adding: kaggle/working/finetune_model_2/generation_config.json (deflated 73%)
  adding: kaggle/working/finetune_model_2/preprocessor_config.json (deflated 42%)


In [34]:
trainer2.evaluate()

{'eval_loss': 0.6318077445030212,
 'eval_wer': 1.2290983606557377,
 'eval_runtime': 178.8144,
 'eval_samples_per_second': 1.722,
 'eval_steps_per_second': 0.218,
 'epoch': 20.0}

## Model 3: learning rate: 1e-5, gradient acculumation steps = 4, num_train_epochs=10

In [35]:
from transformers import WhisperForConditionalGeneration

# Set device

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model3 = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny").to(device)
#model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny")
model3.config.forced_decoder_ids = None
model3.config.suppress_tokens = []

from transformers import Seq2SeqTrainingArguments
training_args3 = Seq2SeqTrainingArguments(
    output_dir="./whisper-tiny-assamese",  # change to a repo name of your choice
    per_device_train_batch_size=16,
    gradient_accumulation_steps = 4,  # increase by 2x for every 2x decrease in batch size
    learning_rate=1e-5,
    warmup_steps=500,
    num_train_epochs=10,
    weight_decay=0.005,
    #max_steps=4000,
    gradient_checkpointing=True,
    fp16=True,
    evaluation_strategy="steps",
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=500,
    eval_steps=50,
    logging_steps=50,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=False,
)

from transformers import Seq2SeqTrainer

trainer3 = Seq2SeqTrainer(
    args=training_args3,
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
)

/opt/conda/lib/python3.10/site-packages/transformers/training_args.py:1545: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [36]:
trainer3.train()
trainer3.save_model("/kaggle/working/finetune_model_3")
!zip -r /kaggle/working/finetune_model_3.zip /kaggle/working/finetune_model_3/

/opt/conda/lib/python3.10/site-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[attr-defined]


Step,Training Loss,Validation Loss,Wer
50,0.143000,0.640930,1.194672
100,0.129500,0.659439,1.270902


/opt/conda/lib/python3.10/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


  adding: kaggle/working/finetune_model_3/ (stored 0%)
  adding: kaggle/working/finetune_model_3/training_args.bin (deflated 51%)
  adding: kaggle/working/finetune_model_3/config.json (deflated 59%)
  adding: kaggle/working/finetune_model_3/model.safetensors (deflated 8%)
  adding: kaggle/working/finetune_model_3/generation_config.json (deflated 73%)
  adding: kaggle/working/finetune_model_3/preprocessor_config.json (deflated 42%)


In [37]:
trainer3.evaluate()

{'eval_loss': 0.6734680533409119,
 'eval_wer': 1.4331967213114754,
 'eval_runtime': 179.4761,
 'eval_samples_per_second': 1.716,
 'eval_steps_per_second': 0.217,
 'epoch': 10.0}

## Model 4: learning rate=1e-5,num_train_steps=5,gradient_acculumation_steps=4 

In [38]:
from transformers import WhisperForConditionalGeneration

# Set device

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model4 = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny").to(device)
#model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny")
model4.config.forced_decoder_ids = None
model4.config.suppress_tokens = []

from transformers import Seq2SeqTrainingArguments
training_args4 = Seq2SeqTrainingArguments(
    output_dir="./whisper-tiny-assamese",  # change to a repo name of your choice
    per_device_train_batch_size=16,
    gradient_accumulation_steps = 4,  # increase by 2x for every 2x decrease in batch size
    learning_rate=1e-5,
    warmup_steps=500,
    num_train_epochs=5,
    weight_decay=0.005,
    #max_steps=4000,
    gradient_checkpointing=True,
    fp16=True,
    evaluation_strategy="steps",
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=500,
    eval_steps=50,
    logging_steps=50,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=False,
)

from transformers import Seq2SeqTrainer

trainer4 = Seq2SeqTrainer(
    args=training_args4,
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
)

/opt/conda/lib/python3.10/site-packages/transformers/training_args.py:1545: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [39]:
trainer4.train()
trainer4.save_model("/kaggle/working/finetune_model_4")
!zip -r /kaggle/working/finetune_model_4.zip /kaggle/working/finetune_model_4/

/opt/conda/lib/python3.10/site-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[attr-defined]


Step,Training Loss,Validation Loss,Wer
50,0.106500,0.681996,1.327869


/opt/conda/lib/python3.10/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


  adding: kaggle/working/finetune_model_4/ (stored 0%)
  adding: kaggle/working/finetune_model_4/training_args.bin (deflated 51%)
  adding: kaggle/working/finetune_model_4/config.json (deflated 59%)
  adding: kaggle/working/finetune_model_4/model.safetensors (deflated 8%)
  adding: kaggle/working/finetune_model_4/generation_config.json (deflated 73%)
  adding: kaggle/working/finetune_model_4/preprocessor_config.json (deflated 42%)


In [40]:
trainer4.evaluate()

{'eval_loss': 0.6832587122917175,
 'eval_wer': 1.2655737704918033,
 'eval_runtime': 175.2716,
 'eval_samples_per_second': 1.757,
 'eval_steps_per_second': 0.223,
 'epoch': 5.0}

## Model 5: learning rate=5e-6 , num_train_epochs=10, gradient_acculumation_steps=1

In [41]:
from transformers import WhisperForConditionalGeneration

# Set device

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model5 = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny").to(device)
#model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny")
model5.config.forced_decoder_ids = None
model5.config.suppress_tokens = []

from transformers import Seq2SeqTrainingArguments
training_args5 = Seq2SeqTrainingArguments(
    output_dir="./whisper-tiny-assamese",  # change to a repo name of your choice
    per_device_train_batch_size=16,
    gradient_accumulation_steps = 1,  # increase by 2x for every 2x decrease in batch size
    learning_rate=5e-6,
    warmup_steps=500,
    num_train_epochs=10,
    weight_decay=0.005,
    #max_steps=4000,
    gradient_checkpointing=True,
    fp16=True,
    evaluation_strategy="steps",
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=500,
    eval_steps=50,
    logging_steps=50,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=False,
)

from transformers import Seq2SeqTrainer

trainer5 = Seq2SeqTrainer(
    args=training_args5,
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
)

/opt/conda/lib/python3.10/site-packages/transformers/training_args.py:1545: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [42]:
trainer5.train()
trainer5.save_model("/kaggle/working/finetune_model_5")
!zip -r /kaggle/working/finetune_model_5.zip /kaggle/working/finetune_model_5/

/opt/conda/lib/python3.10/site-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[attr-defined]


Step,Training Loss,Validation Loss,Wer
50,0.101100,0.686580,1.220902
100,0.099400,0.694983,1.254508
150,0.097500,0.699683,1.238115
200,0.093100,0.709562,1.254918
250,0.086800,0.716835,1.529508
300,0.082400,0.751139,1.247951
350,0.071700,0.768819,1.300820
400,0.066200,0.798443,1.319262
450,0.055000,0.819615,1.218852
500,0.046500,0.835686,1.229918


/opt/conda/lib/python3.10/site-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[attr-defined]
There were missing keys in the checkpoint model loaded: ['proj_out.weight'].
/opt/conda/lib/python3.10/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


updating: kaggle/working/finetune_model_4/ (stored 0%)
updating: kaggle/working/finetune_model_4/training_args.bin (deflated 51%)
updating: kaggle/working/finetune_model_4/config.json (deflated 59%)
updating: kaggle/working/finetune_model_4/model.safetensors (deflated 8%)
updating: kaggle/working/finetune_model_4/generation_config.json (deflated 73%)
updating: kaggle/working/finetune_model_4/preprocessor_config.json (deflated 42%)


In [43]:
trainer5.evaluate()

{'eval_loss': 0.8356863260269165,
 'eval_wer': 1.2299180327868853,
 'eval_runtime': 178.92,
 'eval_samples_per_second': 1.721,
 'eval_steps_per_second': 0.218,
 'epoch': 10.0}

## Model 6: gradient_acculumation_steps=2, lr=1e-5,num_train_epochs=20

In [44]:
from transformers import WhisperForConditionalGeneration

# Set device

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model6 = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny").to(device)
#model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny")
model6.config.forced_decoder_ids = None
model6.config.suppress_tokens = []

from transformers import Seq2SeqTrainingArguments
training_args6 = Seq2SeqTrainingArguments(
    output_dir="./whisper-tiny-assamese",  # change to a repo name of your choice
    per_device_train_batch_size=16,
    gradient_accumulation_steps = 2,  # increase by 2x for every 2x decrease in batch size
    learning_rate=1e-5,
    warmup_steps=500,
    num_train_epochs=20,
    weight_decay=0.005,
    #max_steps=4000,
    gradient_checkpointing=True,
    fp16=True,
    evaluation_strategy="steps",
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=500,
    eval_steps=50,
    logging_steps=50,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=False,
)

from transformers import Seq2SeqTrainer

trainer6 = Seq2SeqTrainer(
    args=training_args6,
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
)

/opt/conda/lib/python3.10/site-packages/transformers/training_args.py:1545: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [45]:
trainer6.train()
trainer6.save_model("/kaggle/working/finetune_model_6")
!zip -r /kaggle/working/finetune_model_6.zip /kaggle/working/finetune_model_6/

/opt/conda/lib/python3.10/site-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[attr-defined]


Step,Training Loss,Validation Loss,Wer
50,0.038200,0.837372,1.205738
100,0.033500,0.866374,1.203279
150,0.028400,0.906941,1.216393
200,0.022200,0.947456,1.104918
250,0.016300,0.984763,1.073361
300,0.014400,1.012029,1.152459
350,0.012400,1.059858,1.043033
400,0.012400,1.060551,1.138525
450,0.011400,1.057214,1.041393
500,0.011200,1.076052,1.063115


/opt/conda/lib/python3.10/site-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[attr-defined]
There were missing keys in the checkpoint model loaded: ['proj_out.weight'].
/opt/conda/lib/python3.10/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


  adding: kaggle/working/finetune_model_6/ (stored 0%)
  adding: kaggle/working/finetune_model_6/training_args.bin (deflated 51%)
  adding: kaggle/working/finetune_model_6/config.json (deflated 59%)
  adding: kaggle/working/finetune_model_6/model.safetensors (deflated 8%)
  adding: kaggle/working/finetune_model_6/generation_config.json (deflated 73%)
  adding: kaggle/working/finetune_model_6/preprocessor_config.json (deflated 42%)


In [ ]:
trainer6.evaluate()

## Model 7: lr=5e-6, gradient_acculumation_steps=4, num_time_epochs=40

In [47]:
from transformers import WhisperForConditionalGeneration

# Set device

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model7 = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny").to(device)
#model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny")
model7.config.forced_decoder_ids = None
model7.config.suppress_tokens = []

from transformers import Seq2SeqTrainingArguments
training_args7 = Seq2SeqTrainingArguments(
    output_dir="./whisper-tiny-assamese",  # change to a repo name of your choice
    per_device_train_batch_size=16,
    gradient_accumulation_steps = 4,  # increase by 2x for every 2x decrease in batch size
    learning_rate=5e-6,
    warmup_steps=500,
    num_train_epochs=40,
    weight_decay=0.005,
    #max_steps=4000,
    gradient_checkpointing=True,
    fp16=True,
    evaluation_strategy="steps",
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=500,
    eval_steps=50,
    logging_steps=50,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=False,
)

from transformers import Seq2SeqTrainer

trainer7 = Seq2SeqTrainer(
    args=training_args6,
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
)

/opt/conda/lib/python3.10/site-packages/transformers/training_args.py:1545: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [48]:
trainer7.train()
trainer7.save_model("/kaggle/working/finetune_model_7")
!zip -r /kaggle/working/finetune_model_7.zip /kaggle/working/finetune_model_7/

/opt/conda/lib/python3.10/site-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[attr-defined]


Step,Training Loss,Validation Loss,Wer
50,0.006500,1.067894,1.183197
100,0.004100,1.092779,1.057787
150,0.003100,1.131710,0.987705
200,0.002400,1.172126,1.020902
250,0.001800,1.219310,0.969672
300,0.002400,1.214146,1.002869
350,0.008400,1.179364,1.181148
400,0.008900,1.128656,1.088525
450,0.008900,1.156736,1.117213
500,0.009400,1.132056,1.010656


/opt/conda/lib/python3.10/site-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[attr-defined]
There were missing keys in the checkpoint model loaded: ['proj_out.weight'].
/opt/conda/lib/python3.10/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


  adding: kaggle/working/finetune_model_7/ (stored 0%)
  adding: kaggle/working/finetune_model_7/training_args.bin (deflated 51%)
  adding: kaggle/working/finetune_model_7/config.json (deflated 59%)
  adding: kaggle/working/finetune_model_7/model.safetensors (deflated 8%)
  adding: kaggle/working/finetune_model_7/generation_config.json (deflated 73%)
  adding: kaggle/working/finetune_model_7/preprocessor_config.json (deflated 42%)


In [49]:
trainer7.evaluate()

{'eval_loss': 1.1320558786392212,
 'eval_wer': 1.0106557377049181,
 'eval_runtime': 166.9229,
 'eval_samples_per_second': 1.845,
 'eval_steps_per_second': 0.234,
 'epoch': 20.0}